# 🗺️ Agent Workflow Memory: Reusing Learned Task Trajectories

## Learning Objectives
In this notebook, you will learn:
1. **Trajectory logging** - record every tool call a ReAct-style loop makes, errors included
2. **Abstraction** - turn a successful trajectory into a template with placeholders
3. **Retrieval by structure** - match a new task to a template by the shape of its steps
4. **Adapt and execute** - fill in the template, resolve step outputs at run time, and record bad steps as errors
5. **Measured payoff** - count LLM calls for planning from scratch vs. adapting a workflow

## Where this fits
`04_voyager_skill_library.ipynb` remembered code. Agent Workflow Memory remembers *procedures* — which tools to call, in which order — one level closer to planning.
The closest idea earlier in the course is module 03's
[`05_long_term_procedural_memory_sqlite.ipynb`](02d_long_term_procedural_memory_sqlite.ipynb): there a
procedure is an instruction set a human approves; here it is a tool-call template the agent induces from its
own successful runs.

## Prerequisites
- No earlier notebook is required: the agent loop is written from scratch here
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`


| Property | Value |
|---|---|
| Origin | Wang et al., *Agent Workflow Memory* (2024). [arXiv:2409.07429](https://arxiv.org/abs/2409.07429) |

## 📖 Definition

**Agent Workflow Memory (AWM)** is a memory architecture where an agent remembers and reuses entire
**multi-step procedures** (sequences of tool calls) that previously solved a task successfully, rather
than remembering isolated facts (episodic/semantic memory) or standalone executable code skills
(as in Voyager-style skill libraries).

The core idea, inspired by the `35_agent_workflow_memory` concept in
[FareedKhan-dev/all-agentic-architectures](https://github.com/FareedKhan-dev/all-agentic-architectures)
(re-implemented here from scratch, not copied), is:

> If a task the agent just solved is *structurally similar* to a future task (same shape, different
> specific values), don't re-plan from scratch — retrieve the workflow **template** that worked before,
> **adapt** its placeholders to the new task, and execute it directly.

### High-Level Workflow

1. **Act** — run the agent on a task with a small ReAct-ish loop (pick tool -> call tool -> observe -> repeat).
2. **Log the trajectory** — record the exact sequence of tool calls (name + arguments) that led to success.
3. **Abstract** — an LLM call turns the concrete trajectory into a generalized workflow template, replacing
   specific values with placeholders (e.g. `"1. look up {entity}'s value  2. multiply by {factor}"`).
4. **Store** — save `{task_description, workflow_steps, success}` in a workflow memory store.
5. **Retrieve & adapt** — for a new task, search the store for the most structurally similar remembered
   task. If found, ask the LLM to *fill in* the template's placeholders for the new task instead of
   planning from a blank slate, then execute the filled-in steps directly.
6. **Fallback** — if no similar workflow exists, fall back to normal from-scratch ReAct planning.

### When to Use

- Agents that face **repeated task families** with the same shape but different concrete values
  (e.g. "look up X, then compute Y from it" across many different X/Y).
- Environments with a stable, small tool surface, where the *sequence* of tool calls (not the code)
  is the reusable unit.
- Systems where reducing planning latency/tokens on recurring task shapes matters (agentic copilots,
  workflow automation, repeated data-lookup-and-compute pipelines).

### Strengths & Weaknesses

**Strengths**

- **Faster & cheaper** on recurring task shapes — adapting a template skips exploratory planning.
- **More consistent** — a workflow that worked before is less likely to go off-track than a fresh plan.
- **Complementary** to episodic/semantic memory (facts) and skill libraries (code) — it captures the
  *procedure*, sitting one level of abstraction above both.

**Weaknesses**

- Templates can **overfit** to the surface form of the first task seen — retrieval must judge structural
  similarity, not just keyword overlap, or it will mis-apply a workflow to a task it doesn't actually fit.
- Requires a **success signal** to know which trajectories are worth abstracting and storing.
- Doesn't help with genuinely **novel** task shapes — those always fall back to full planning.
- Templates can go **stale** if the tools or their semantics change.

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: plans, abstracts, retrieves and adapts workflows
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# IMPORTS AND AN LLM-CALL COUNTER
# ============================================================================
import json
import re
from typing import Any

LLM_CALLS = {"count": 0}


def call_llm(messages):
    """Invoke the model and count the call, so we can compare planning strategies by cost."""
    LLM_CALLS["count"] += 1
    return llm.invoke(messages)


def parse_json(raw: str):
    """Parse model output as JSON, tolerating a markdown code fence."""
    raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.IGNORECASE).strip()
    return json.loads(raw)


print("✅ Helpers ready")

---
## 🧰 Part 1: Tools and Trajectory Logging

### 1.1 🧮 A Small Toolset

We use two tools:

- **`lookup(entity)`** — looks up a numeric value for an entity in a small in-notebook "database"
  (stand-in for e.g. a product catalog, a stock price table, an inventory system).
- **`calculator(expression)`** — evaluates `+ - * /` arithmetic. Exponentiation is refused: `9**9**9**9` is a
  short string whose value would exhaust memory.

Every call goes through `call_tool`, which turns a bad call — an unknown tool, wrong argument names — into an
`ERROR:` observation the agent can read, instead of an exception that ends the run.

Structurally similar tasks will look like: *"look up {entity}'s value, then do some arithmetic with it"*.


In [ ]:
# ============================================================================
# PART 1.1: TOOLS, AND ONE PLACE THAT CALLS THEM
# ============================================================================
DATASET = {
    "widget_a": 12,
    "widget_b": 7,
    "widget_c": 25,
    "gadget_x": 40,
    "gadget_y": 3,
}


def lookup(entity: str) -> str:
    """Look up the numeric value associated with an entity in the dataset."""
    key = entity.strip().lower().replace(" ", "_")
    if key not in DATASET:
        return f"ERROR: '{entity}' not found in dataset."
    return str(DATASET[key])


def calculator(expression: str) -> str:
    """Evaluate + - * / arithmetic, e.g. '12 * 4'. No exponentiation."""
    # Digits, + - * /, spaces, parentheses and decimal points only -- and no '**' (9**9**9**9 exhausts memory)
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression) or "**" in expression:
        return f"ERROR: unsafe expression '{expression}'."
    try:
        return str(eval(expression, {"__builtins__": {}}))  # noqa: S307 - only arithmetic characters get here
    except Exception as exc:
        return f"ERROR: {exc}"


TOOLS = {
    "lookup": lookup,
    "calculator": calculator,
}

TOOL_DESCRIPTIONS = """\
- lookup(entity: str) -> str : look up a numeric value for an entity in the dataset
- calculator(expression: str) -> str : evaluate + - * / arithmetic (no exponentiation)
"""


def call_tool(name: str, args) -> str:
    """Run one tool call; return problems as an ERROR observation instead of raising."""
    tool_fn = TOOLS.get(name)
    if tool_fn is None:
        return f"ERROR: unknown tool '{name}'"
    if not isinstance(args, dict):
        return f"ERROR: arguments must be a JSON object, got {args!r}"
    try:
        return tool_fn(**args)
    except TypeError as exc:  # wrong or missing argument names, e.g. lookup(name=...)
        return f"ERROR: bad arguments for {name}: {exc}"


print(call_tool("calculator", {"expression": "12 * 4"}))
print(call_tool("calculator", {"expression": "9**9**9**9"}))
print(call_tool("lookup", {"name": "widget_a"}))

### 1.2 🗃️ The Workflow Memory Store

Each record is `{task_description, workflow_steps, success}`, where `workflow_steps` is a list of
human-readable, **generalized** step strings (placeholders like `{entity}`, `{factor}` instead of
concrete values). This is intentionally simple — a list in memory — to keep the retrieval logic
transparent; a production system might back this with a vector store keyed on `task_description`.


In [ ]:
# ============ WORKFLOW MEMORY STORE ============
workflow_memory: list[dict[str, Any]] = []


def add_workflow(task_description: str, workflow_steps: list[str], success: bool) -> None:
    workflow_memory.append(
        {
            "task_description": task_description,
            "workflow_steps": workflow_steps,
            "success": success,
        }
    )


def show_workflow_memory() -> None:
    for i, wf in enumerate(workflow_memory):
        print(f"[{i}] task: {wf['task_description']!r}  success={wf['success']}")
        for step in wf["workflow_steps"]:
            print(f"      - {step}")


print("✅ Workflow memory ready (empty)")

### 1.3 🔁 A Minimal ReAct-ish Agent Loop (with Trajectory Logging)

This is a deliberately simple loop: at each step we ask the LLM to either call a tool (as JSON) or
declare `"final_answer"`. Every tool call the agent actually makes is appended to `trajectory` —
this concrete, executed sequence is what we'll later abstract into a workflow template.


In [ ]:
# ============================================================================
# PART 1.3: REACT-ISH LOOP WITH TRAJECTORY LOGGING
# ============================================================================
REACT_SYSTEM_PROMPT = f"""You are a task-solving agent with access to these tools:
{TOOL_DESCRIPTIONS}
At each turn, respond with ONLY a JSON object, no extra text, in one of these two forms:
1. To call a tool:      {{"action": "tool", "name": "<tool_name>", "args": {{...}}}}
2. To give final answer: {{"action": "final_answer", "value": "<answer>"}}
Example tool call: {{"action": "tool", "name": "calculator", "args": {{"expression": "7 * 2"}}}}

Use the tool observations you've been given so far to decide your next step.
Keep the trajectory as short as possible.
"""


def run_agent(task: str, max_steps: int = 5) -> tuple[str, list[dict]]:
    """Run the ReAct-ish loop from scratch (blank-slate planning). Returns (final_answer, trajectory)."""
    trajectory: list[dict] = []
    history = f"Task: {task}\n"
    for _ in range(max_steps):
        response = call_llm(
            [
                {"role": "system", "content": REACT_SYSTEM_PROMPT},
                {"role": "user", "content": history},
            ]
        )
        raw = response.content.strip()
        try:
            step = parse_json(raw)
            if not isinstance(step, dict):
                raise json.JSONDecodeError("not a JSON object", raw, 0)
        except json.JSONDecodeError:
            history += f"\n(Could not parse response: {raw!r}. Respond with valid JSON only.)\n"
            continue

        if step.get("action") == "final_answer":
            return step.get("value", ""), trajectory

        if step.get("action") == "tool":
            name, args = step.get("name"), step.get("args", {})
            observation = call_tool(name, args)  # problems come back as observations, not exceptions
            trajectory.append({"tool": name, "args": args, "observation": observation})
            history += f"\nCalled {name}({args}) -> {observation}\n"
        else:  # valid JSON in the wrong shape: say so, or the model repeats it every step
            history += f"\n(Unrecognised step {raw!r}. Use one of the two JSON forms above.)\n"

    return "FAILED: max steps reached", trajectory


print("✅ Agent loop ready")

---
## 🧩 Part 2: Abstracting a Trajectory into a Workflow Template

Once a task succeeds, we ask the LLM to look at the **concrete** trajectory and rewrite it as a
**generalized** step list, replacing specific values (entity names, numbers) with placeholders like
`{entity}` and `{factor}`. This generalized template is what gets stored in `workflow_memory` — not
the raw trajectory.


In [ ]:
# ============ ABSTRACTION: TRAJECTORY -> GENERALIZED WORKFLOW TEMPLATE ============
ABSTRACTION_PROMPT = """You just watched an agent solve a task using this sequence of tool calls:

Task: {task}

Trajectory:
{trajectory}

Rewrite this as a SHORT, GENERALIZED workflow template: a numbered list of steps describing the
PROCEDURE in the abstract, replacing concrete values (entity names, numbers, factors) with
placeholders like {{entity}}, {{factor}}, {{amount}}. Do not include the concrete values themselves.

Respond with ONLY a JSON list of strings, e.g.:
["1. look up {{entity}}'s value", "2. multiply the result by {{factor}}"]
"""


def abstract_trajectory(task: str, trajectory: list[dict]) -> list[str]:
    trajectory_str = "\n".join(
        f"- {t['tool']}({t['args']}) -> {t['observation']}" for t in trajectory
    )
    response = call_llm(
        [{"role": "user", "content": ABSTRACTION_PROMPT.format(task=task, trajectory=trajectory_str)}]
    )
    raw = response.content.strip()
    try:
        return parse_json(raw)
    except json.JSONDecodeError:
        # Fall back to a naive split if the LLM didn't return clean JSON.
        return [line.strip("- ") for line in raw.splitlines() if line.strip()]


print("✅ Abstraction ready")

In [ ]:
# ============ RUN THE AGENT ON A FIRST TASK & LEARN A WORKFLOW ============
task_1 = "Look up widget_a's value and multiply it by 3."
answer_1, trajectory_1 = run_agent(task_1)

print("Task:", task_1)
print("Answer:", answer_1)
print("Trajectory:", trajectory_1)


In [ ]:
# ============================================================================
# ABSTRACT THE SUCCESSFUL TRAJECTORY AND STORE IT
# ============================================================================
success_1 = answer_1 not in ("", None) and not str(answer_1).startswith("FAILED")

if success_1:
    template_1 = abstract_trajectory(task_1, trajectory_1)
    add_workflow(task_1, template_1, success=True)

show_workflow_memory()


### Discussion of the Output

The agent solved `task_1` from a blank slate: two tool calls (`lookup` then `calculator`), each
decided one step at a time by the LLM re-reasoning about the whole history so far. After success,
the abstraction step compressed that concrete trajectory into a generalized 2-step template with
`{entity}` and `{factor}` placeholders. That template — not the raw trajectory — is what now lives
in `workflow_memory`, ready to be reused for a differently-worded but structurally identical task.


In [ ]:
# ============ LEARN A SECOND, DIFFERENTLY-SHAPED WORKFLOW ============
task_2 = "Look up gadget_x's value, then look up gadget_y's value, and add them together."
answer_2, trajectory_2 = run_agent(task_2)

print("Task:", task_2)
print("Answer:", answer_2)
print("Trajectory:", trajectory_2)

success_2 = answer_2 not in ("", None) and not str(answer_2).startswith("FAILED")
if success_2:
    template_2 = abstract_trajectory(task_2, trajectory_2)
    add_workflow(task_2, template_2, success=True)

show_workflow_memory()


We now have **two** distinct workflow templates in memory:

1. `lookup(entity) -> calculator(value * factor)` — a "look up then scale" shape.
2. `lookup(entity_1) -> lookup(entity_2) -> calculator(sum)` — a "look up two things then combine" shape.

This diversity is what makes retrieval meaningful in the next step: a new task needs to be matched
against the *right* template, not just any template.


---
## 🔍 Part 3: Retrieving a Workflow by Structure

For a new task, we ask the LLM to judge which stored `task_description` (if any) is **structurally
similar** to the new task — same shape of steps, regardless of the specific entities/numbers involved.
This is a lightweight LLM-similarity match rather than embedding search, which keeps the notebook
self-contained, but a production system would typically embed `task_description` and use a vector
store for this lookup.


In [ ]:
# ============ RETRIEVE THE CLOSEST MATCHING WORKFLOW ============
RETRIEVAL_PROMPT = """You are matching a new task against a library of remembered workflows, by
STRUCTURE (the shape/sequence of steps), not by the specific values mentioned.

New task: {new_task}

Remembered workflows:
{workflows}

If one of the remembered workflows has the SAME STRUCTURE as the new task (same kind of steps in the
same order, just with different specific values), respond with ONLY its index number (e.g. "0").
If none of them match structurally, respond with ONLY the word "NONE".
"""


def retrieve_workflow(new_task: str) -> dict | None:
    if not workflow_memory:
        return None
    workflows_str = "\n".join(
        f"[{i}] {wf['task_description']} -> steps: {wf['workflow_steps']}"
        for i, wf in enumerate(workflow_memory)
    )
    response = call_llm(
        [{"role": "user", "content": RETRIEVAL_PROMPT.format(new_task=new_task, workflows=workflows_str)}]
    )
    raw = response.content.strip()
    if raw.upper().startswith("NONE"):
        return None
    try:
        idx = int(re.search(r"\d+", raw).group())
        return workflow_memory[idx]
    except (AttributeError, ValueError, IndexError):
        return None


print("✅ Workflow retrieval ready")

---
## 🛠️ Part 4: Adapting a Retrieved Workflow

Instead of planning from scratch, we hand the LLM the **retrieved template** and the **new task**,
and ask it to fill in the placeholders with concrete values and produce concrete tool calls to
execute — directly, without the step-by-step exploratory reasoning `run_agent` does.
The executor resolves each `{step_N}` from earlier observations. A plan can also be wrong — a step that refers to
`{step_3}` in a two-step plan, or an argument name the tool doesn't have — and the executor records that as an
`ERROR:` observation instead of crashing.

In [ ]:
# ============================================================================
# ADAPT & EXECUTE A RETRIEVED WORKFLOW
# ============================================================================
# Built with .replace + one .format pass: literal JSON braces are doubled exactly once.
# (An f-string followed by .format() would un-double them twice and crash on the JSON example.)
ADAPTATION_PROMPT = """You are given a generalized workflow template and a new task. Fill in the
template's placeholders with concrete values from the new task, and produce the CONCRETE sequence of
tool calls needed to execute it.

Available tools:
<<TOOLS>>
Template steps: {template}
New task: {new_task}

You do NOT know the looked-up values yet. In a later step's arguments, write {{step_1}}, {{step_2}}, ...
to use the observation returned by that earlier step.

Respond with ONLY a JSON list of tool calls, e.g.:
[{{"name": "lookup", "args": {{"entity": "widget_b"}}}}, {{"name": "calculator", "args": {{"expression": "{{step_1}} * 2"}}}}]
""".replace("<<TOOLS>>", TOOL_DESCRIPTIONS)


def resolve_refs(value, trajectory: list[dict]):
    """Replace {step_N} with step N's observation; raise LookupError for a step that hasn't run."""
    if not isinstance(value, str):
        return value

    def observation(match):
        n = int(match[1])
        if not 1 <= n <= len(trajectory):
            raise LookupError(f"{{step_{n}}} refers to a step that has not run ({len(trajectory)} so far)")
        return trajectory[n - 1]["observation"]

    return re.sub(r"\{step_(\d+)\}", observation, value)


def execute_plan(plan: list) -> tuple[str | None, list[dict]]:
    """Run a concrete plan in order; a bad step reference or tool call is recorded as an ERROR observation."""
    trajectory: list[dict] = []
    for step in plan:
        if not isinstance(step, dict):
            trajectory.append({"tool": None, "args": step, "observation": "ERROR: a plan step must be a JSON object"})
            continue
        name, args = step.get("name"), step.get("args", {})
        try:
            if isinstance(args, dict):
                args = {key: resolve_refs(value, trajectory) for key, value in args.items()}
            observation = call_tool(name, args)
        except LookupError as exc:
            observation = f"ERROR: {exc}"
        trajectory.append({"tool": name, "args": args, "observation": observation})
    return (trajectory[-1]["observation"] if trajectory else None), trajectory


def adapt_and_execute(template_workflow: dict, new_task: str) -> tuple[str | None, list[dict]]:
    response = call_llm(
        [{"role": "user", "content": ADAPTATION_PROMPT.format(template=template_workflow["workflow_steps"], new_task=new_task)}]
    )
    try:
        plan = parse_json(response.content)
    except json.JSONDecodeError:
        plan = []
    return execute_plan(plan if isinstance(plan, list) else [])


print("✅ Adapter and plan executor ready")

In [ ]:
# ============================================================================
# A BROKEN PLAN IS RECORDED, NOT RAISED
# ============================================================================
broken_plan = [
    {"name": "lookup", "args": {"name": "widget_a"}},               # wrong argument name
    {"name": "lookup", "args": {"entity": "widget_a"}},
    {"name": "calculator", "args": {"expression": "{step_5} * 2"}},  # refers to a step that never runs
]
answer_broken, trajectory_broken = execute_plan(broken_plan)
for i, step in enumerate(trajectory_broken, 1):
    print(f"step {i}: {step['tool']}({step['args']}) -> {step['observation']}")

errors = sum(step["observation"].startswith("ERROR") for step in trajectory_broken)
print(f"\n{'✅' if errors == 2 and len(trajectory_broken) == 3 else '❌'} {errors} errors recorded, and all 3 steps ran")

---
## 📏 Part 5: Adapting vs. Planning From Scratch

Now the payoff. A new task with the same shape as `task_1` is solved twice: once by retrieving and adapting the
remembered workflow, once by the from-scratch loop. Both answers are checked against the value computed in code,
and both LLM call counts are printed.

In [ ]:
# ============================================================================
# NEW, STRUCTURALLY-SIMILAR TASK: ADAPT INSTEAD OF PLAN, AND COMPARE COSTS
# ============================================================================
new_similar_task = "Look up widget_c's value and multiply it by 4."
expected = DATASET["widget_c"] * 4  # ground truth, computed in code

LLM_CALLS["count"] = 0
retrieved = retrieve_workflow(new_similar_task)
print("Retrieved template:", retrieved["workflow_steps"] if retrieved else None)

if retrieved:
    answer_adapted, trajectory_adapted = adapt_and_execute(retrieved, new_similar_task)
    adapted_calls = LLM_CALLS["count"]
    print("\n--- Solved by ADAPTING a remembered workflow ---")
    print("Trajectory:", trajectory_adapted)
    print(f"Answer: {answer_adapted}  (expected {expected}) -> {'✅ correct' if str(answer_adapted).strip() in (str(expected), f'{expected}.0') else '❌ wrong'}")
    print(f"LLM calls (retrieve + adapt): {adapted_calls}")
else:
    print("No matching workflow found - would fall back to run_agent().")

# The same task planned from scratch, for comparison
LLM_CALLS["count"] = 0
answer_scratch, _ = run_agent(new_similar_task)
print(f"\n--- The same task planned from scratch ---\nAnswer: {answer_scratch}\nLLM calls: {LLM_CALLS['count']}")

### Discussion of the Output

Compare this to how `task_1` was solved earlier: there, the agent took multiple LLM calls, each one
re-reading the whole running history and deciding the *next single step*. Here, retrieval matched
`new_similar_task` to template `[0]` (the "look up then scale" shape) in one LLM call, and
`adapt_and_execute` produced the **entire concrete tool-call plan in a single shot**, then executed
it directly — no exploratory step-by-step reasoning needed. This is the speed/directness gain AWM is
meant to provide on recurring task shapes: fewer LLM round-trips, and a plan that mirrors a
previously-successful procedure instead of being re-derived from zero.

The printed call counts make the comparison concrete: adapting costs a fixed two calls (retrieve + adapt) however many steps the workflow has, while from-scratch planning costs one call per step plus the final answer. The `{step_1}` reference is what lets a one-shot plan use a value it has not looked up yet — the executor fills it in, and the answer is checked against the value computed in code.

### 5.1 🆕 A Novel Task: No Matching Workflow, Fall Back to Planning

Now let's give the agent a task with a genuinely different shape — one that doesn't match either
stored template. Retrieval should return `None`, and we fall back to full `run_agent` planning,
exactly as we did for `task_1` and `task_2` before anything was in memory.


In [ ]:
# ============ NOVEL TASK: NO WORKFLOW MATCH -> FALL BACK TO FROM-SCRATCH PLANNING ============
novel_task = "What is 17 times 5, minus 9?"

retrieved_novel = retrieve_workflow(novel_task)
print("Retrieved template for novel task:", retrieved_novel)
print(f"{'✅' if retrieved_novel is None else '❌'} no remembered workflow matched the novel task")

if retrieved_novel is None:
    print("\n--- No matching workflow: falling back to blank-slate ReAct planning ---")
    answer_novel, trajectory_novel = run_agent(novel_task)
    print("Task:", novel_task)
    print("Trajectory:", trajectory_novel)
    print(f"Answer: {answer_novel}  (expected {17 * 5 - 9})")


### Discussion of the Output

`novel_task` doesn't involve a `lookup` at all — it's pure arithmetic, structurally unlike either
remembered workflow (which both start with one or more `lookup` calls). Retrieval returns `None` — the check
line above confirms it — and the agent falls back to the same step-by-step `run_agent` loop used before any workflow
memory existed. This is the expected trade-off: AWM accelerates *recurring* task shapes, but a
genuinely novel shape still requires full planning — and, if it succeeds, could itself be abstracted
into a *third* workflow template for next time.


---
## 📝 Summary

In this notebook, we learned:

### 1. Trajectory logging
- A small ReAct-ish loop records every tool call it makes; bad calls (unknown tools, wrong argument names) and unsafe arithmetic come back as `ERROR:` observations instead of ending the run

### 2. Abstraction
- After a success, an LLM call turns the concrete trajectory into a generalized template with placeholders like `{entity}` and `{factor}` — the template, not the raw trajectory, is what gets stored

### 3. Retrieval by structure
- A new task is matched to a template by the *shape* of its steps, not by keyword overlap; we used an LLM-similarity check, while a production system would typically embed `task_description` and use a vector store

### 4. Adapt and execute
- The LLM fills in the template's placeholders in one shot; `{step_N}` references are resolved from earlier observations at run time, and a reference to a step that never ran is recorded as an error

### 5. Measured payoff
- Adapting cost a fixed two LLM calls (retrieve + adapt); planning from scratch cost one call per step plus the final answer. A genuinely novel task matches no workflow and falls back to planning

**How this differs from Voyager-style skill libraries:** Voyager stores standalone, directly executable **code
skills** that can be called like black-box subroutines. AWM instead stores **procedural knowledge about which tools
to call, in which order, for which kind of task** — a workflow template still needs the agent's tool-calling loop to
execute it (each step is adapted and dispatched to a real tool). AWM is one level closer to "how the agent should
plan", where Voyager is one level closer to "new capabilities the agent can invoke".

### Next Steps
- **Module 05 — `05_multi_agent_foundations`**: from one agent's memory to several agents working together